# First-generation attacks — the workshop where the attack zoo was built

Before there was a library, there was this notebook: the attacks developed and tuned one by one against the DNN detector on the **redefined feature matrix**. The defining trait of this generation is what it does *not* have yet — no insertion/removal constraints; the only nod to the binary domain is **rounding** the continuous perturbation back to 0/1.

Reading order inside: rounding first (everything depends on it), then the k-step gradient family (dFGSM-k, BGA-k, BCA-k, Grosse-k) with their parameter sweeps and success accounting, PGD, the metrics harness, and finally the max family (PGD-Max, StepwiseMax and its one-step variants) and mimicry. The frozen versions of these functions live in `attack_library.ipynb`.

**Where this fits:** Section 03 attacks the detectors built in `02-detectors/`, under the manipulation rules from `01-data-and-features/manipulation_maps.ipynb`. The defended models these attacks are later measured against live in `04-defenses/`, and the final, organised measurements are in `06-thesis-evaluation/`.

*Consolidated from `3_adverserial_attacks.ipynb` (originally in `a_malware2`). The notebook's own markdown fragments are kept where they explain a step. Outputs cleared.*

## Step 1 — Setup: the redefined matrix and the trained detector

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from scipy import sparse
import gdown
import numpy as np
import matplotlib.pyplot as plt
import pickle
import torch.nn.functional as F
import random

In [ ]:
download_link = 'https://drive.google.com/uc?id=12iud4h19CZst4exbr3U2A9iDxBYvZ5U_'
output_filepath = '/content/'
gdown.download(download_link, output_filepath)

In [ ]:
download_link = 'https://drive.google.com/uc?id=1IhrcT3jHqlPrw2KvQ5vJkBgozxcJ1cJm'
output_filepath = '/content/'
gdown.download(download_link, output_filepath)

In [ ]:
download_link = 'https://drive.google.com/uc?id=13o5n06UpMDOhtk4u7B_RBSWa3kiiGXFs'
output_filepath = '/content/'
gdown.download(download_link, output_filepath)

In [ ]:
download_link = 'https://drive.google.com/uc?id=1PxFOLBnQAlX-EOsqkhGCSd1T3ykAD0-4'
output_filepath = '/content/'
gdown.download(download_link, output_filepath)

In [ ]:
# Load the dictionary from the file
with open('vocab.pkl', 'rb') as f:
    vocab = pickle.load(f)

for i, (key, value) in enumerate(vocab.items()):
    print((key, value))
    if i >= 5:
        break

In [ ]:
# Load dataset
X_redefined = sparse.load_npz("X_redefined_sparse_matrix.npz")
labels_tensor = torch.load('labels.pt')

In [ ]:
# Split data into train, validation, and test sets with stratified sampling
X_train_val, X_test, labels_train_val, labels_test = train_test_split(X_redefined, labels_tensor, test_size=0.2, stratify=labels_tensor, random_state=42)
X_train, X_val, labels_train, labels_val = train_test_split(X_train_val, labels_train_val, test_size=0.2, stratify=labels_train_val, random_state=42)

# Combine features and labels into datasets
train_dataset = TensorDataset(torch.tensor(X_train.toarray(), dtype=torch.float32), labels_train)
val_dataset = TensorDataset(torch.tensor(X_val.toarray(), dtype=torch.float32), labels_val)
test_dataset = TensorDataset(torch.tensor(X_test.toarray(), dtype=torch.float32), labels_test)

In [ ]:
# Define the DataLoader for training, validation, and test sets
batch_size = 256
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
import torch
import torch.nn as nn

class MalwareDetectionModel(nn.Module):
    def __init__(self, input_size=10000, hidden_1_size=200, hidden_2_size=200, num_labels=2, dropout_prob=0.6):
        super(MalwareDetectionModel, self).__init__()

        self.input_size = input_size
        self.hidden_1_size = hidden_1_size
        self.hidden_2_size = hidden_2_size
        self.num_labels = num_labels
        self.dropout_prob = dropout_prob

        self.fc1 = nn.Linear(input_size, hidden_1_size)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout_prob)
        self.fc2 = nn.Linear(hidden_1_size, hidden_2_size)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout_prob)
        self.fc3 = nn.Linear(hidden_2_size, num_labels)
        self.log_softmax = nn.LogSoftmax(dim=1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        x = self.log_softmax(x)
        return x

In [ ]:
# Create an instance of your model
model = MalwareDetectionModel()

# Load model parameters
model.load_state_dict(torch.load('DNN_params.pth'))

In [ ]:
X,y = next(iter(test_loader))
print(X.shape)
print(y.shape)

## Step 2 — Rounding: back from continuous perturbations to binary features



```
delta = torch.zeros_like(X, requires_grad=True)
for t in range(25):
    loss = nn.CrossEntropyLoss()(model(X + delta), y.view(-1).long())
    loss.backward()
    gradients = delta.grad.detach().sign() * (X < 0.5)
    delta.data = (delta + 0.02*delta.grad.detach().sign()).clamp(0.,1.)
    print(delta.data[0])
    delta.grad.zero_()
```



In [ ]:
def round_x(x, round_threshold=0.5):
    """
    Rounds x by thresholding it according to round_threshold.
    :param x: input tensor
    :param round_threshold: threshold parameter
    :return: a tensor of 0s and 1s
    """
    return (x >= round_threshold).float()

def get_x0(x, initial_rounding_threshold=0.5, is_sample=False):
    """
    Helper function to randomly initialize the inner maximizer algorithm.
    Randomizes the input tensor while preserving its functionality.
    :param x: input tensor
    :param rounding_threshold: threshold for rounding
    :param is_sample: flag to sample randomly from feasible area
    :return: randomly sampled feasible version of x
    """
    if is_sample:
        rand_x = round_x(torch.rand(x.size()), initial_rounding_threshold=initial_rounding_threshold)
        return (rand_x.byte() | x.byte()).float()
    else:
        return x

def or_float_tensors(x_1, x_2):
    """
    ORs two float tensors by converting them to byte and back.
    :param x_1: tensor one
    :param x_2: tensor two
    :return: float tensor of 0s and 1s
    """
    return (x_1.byte() | x_2.byte()).float()


def xor_float_tensors(x_1, x_2):
    """
    XORs two float tensors by converting them to byte and back
    Note that byte() takes the first 8 bit after the decimal point of the float
    e.g., 0.0 ==> 0
          0.1 ==> 0
          1.1 ==> 1
        255.1 ==> 255
        256.1 ==> 0
    Subsequently the purpose of this function is to map 1s float tensors to 1
    and those of 0s to 0. I.e., it is meant to be used on tensors of 0s and 1s.

    :param x_1: tensor one
    :param x_2: tensor two
    :return: float tensor of 0s and 1s.
    """
    return (x_1.byte() ^ x_2.byte()).float()

def get_loss(x,y,model):
    criterion = nn.CrossEntropyLoss(reduction='none')
    outputs = model(x)
    loss = criterion(outputs, y.view(-1).long())
    _, predicted = torch.topk(outputs, k=1)
    done = (predicted != y).squeeze()

    return loss, done



```
x_1 = torch.tensor([1.0, 1.0,  1.0, 1.0, 1.0, 0.0,  0.0, 0.2])
x_2 = torch.tensor([0.0, 0.6, -0.2, 0.9, 1.9, 1.0, -0.2, 0.3])
result = or_float_tensors(x_1, x_2)
             tensor([1., 1., 1., 1., 1., 1., 0., 0.])

```



## Step 3 — dFGSM-k, and its second version

In [ ]:
def dfgsm_k(x, y, model, k=25, epsilon=0.02, alpha=1., initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    FGSM^k with deterministic rounding
    :param y: ground truth labels
    :param x: feature vector
    :param model: neural network model
    :param k: number of steps
    :param epsilon: update value in each direction
    :param alpha: hyperparameter for controlling the portionate of rounding
    :param initial_rounding_threshold: threshold parameter for rounding the initial x_next
    :param round_threshold: threshold parameter for rounding
    :param is_report_loss_diff: flag to report loss difference
    :param is_sample: flag to sample randomly from the feasible area
    :return: the adversarial version of x according to dfgsm_k (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), y.view(-1).long())

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)

    # Multi-step
    for t in range(k):
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, y.view(-1).long())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        # Find the next sample
        x_next = x_next + epsilon * torch.sign(grad_vars[0].data)

        # Projection
        x_next = torch.clamp(x_next, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size()) * alpha
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Feasible projection
    x_next = or_float_tensors(x_next, x)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), y.view(-1).long()).data

    if is_report_loss_diff:
        #print(f"Natural loss: {loss_natural.mean():.4f}, Adversarial loss: {loss_adv.mean():.4f}, Difference: {(loss_adv.mean() - loss_natural.mean()):.4f}")
        outputs = model(x_next)
        _, predicted = torch.topk(outputs, k=1)
        done = (predicted != y).squeeze()
        print(f"dFGSM: attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv < loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    return x_next

In [ ]:
x_adv = dfgsm_k(X,y,model)
print(X.sum()/batch_size)
print(x_adv.sum()/batch_size)

In [ ]:
x_adv = dfgsm_k(X,y,model,random=True)
print(X.sum()/batch_size)
print(x_adv.sum()/batch_size)

In [ ]:
x_adv = dfgsm_k(X,y,model,k=1,random=True)
print(X.sum()/batch_size)
print(x_adv.sum()/batch_size)

In [ ]:
for alpha in range(1, 52, 10):
  print('alpha = ',alpha)
  x_adv = dfgsm_k(X, y, model, k=20, epsilon=0.02, alpha=alpha,random=True)
  print(X.sum()/batch_size)
  print(x_adv.sum()/batch_size)
  print('***************************')

In [ ]:
total_samples = 0
total_attack_success = 0
total_features_x = 0
total_features_x_adv = 0

for X, y in test_loader:
    batch_size = X.size(0)
    x_adv = dfgsm_k(X, y, model, k=20, epsilon=0.0005, random=True,is_report_loss_diff=False)

    # Compute mean number of features
    total_features_x += X.sum().item()
    total_features_x_adv += x_adv.sum().item()

    # Compute attack effectiveness for this batch
    outputs_adv = model(x_adv)
    _, predicted_adv = torch.topk(outputs_adv, k=1)
    total_attack_success += torch.sum(predicted_adv != y).item()
    total_samples += batch_size

# Compute mean number of features
mean_features_x = total_features_x / total_samples
mean_features_x_adv = total_features_x_adv / total_samples

# Compute mean attack effectiveness
mean_attack_effectiveness = (total_attack_success / total_samples) * 100

# Print results
print('Mean number of features (X):', mean_features_x)
print('Mean number of features (X_adv):', mean_features_x_adv)
print('Mean attack effectiveness:', mean_attack_effectiveness)

In [ ]:
def dfgsm_k2(x, y, model, k=25, epsilon=0.02, alpha=1., initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    FGSM^k with deterministic rounding
    :param y: ground truth labels
    :param x: feature vector
    :param model: neural network model
    :param k: number of steps
    :param epsilon: update value in each direction
    :param alpha: hyperparameter for controlling the portionate of rounding
    :param initial_rounding_threshold: threshold parameter for rounding the initial x_next
    :param round_threshold: threshold parameter for rounding
    :param is_report_loss_diff: flag to report loss difference
    :param is_sample: flag to sample randomly from the feasible area
    :return: the adversarial version of x according to dfgsm_k (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), y.view(-1).long())

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)

    # Multi-step
    for t in range(k):
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, y.view(-1).long())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        #print(torch.sign(grad_vars[0].data).clamp(min=0))
        # Find the next sample
        x_next = x_next + epsilon * (torch.sign(grad_vars[0].data).clamp(min=0))

        # Projection
        x_next = torch.clamp(x_next, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size()) * alpha
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Feasible projection
    x_next = or_float_tensors(x_next, x)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), y.view(-1).long()).data

    if is_report_loss_diff:
        #print(f"Natural loss: {loss_natural.mean():.4f}, Adversarial loss: {loss_adv.mean():.4f}, Difference: {(loss_adv.mean() - loss_natural.mean()):.4f}")
        outputs = model(x_next)
        _, predicted = torch.topk(outputs, k=1)
        done = (predicted != y).squeeze()
        print(f"rFGSM: attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv < loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    return x_next

In [ ]:
x_adv = dfgsm_k2(X[:1],y[:1],model)

In [ ]:
for alpha in range(1, 52, 10):
  print('alpha = ',alpha)
  x_adv = dfgsm_k2(X, y, model, k=20, epsilon=0.02, alpha=alpha,random=True)
  print(X.sum()/batch_size)
  print(x_adv.sum()/batch_size)
  print('***************************')

In [ ]:
total_samples = 0
total_attack_success = 0
total_features_x = 0
total_features_x_adv = 0

for X, y in test_loader:
    batch_size = X.size(0)
    x_adv = dfgsm_k2(X, y, model, k=20, epsilon=0.02, alpha=40,random=True,is_report_loss_diff=False)

    # Compute mean number of features
    total_features_x += X.sum().item()
    total_features_x_adv += x_adv.sum().item()

    # Compute attack effectiveness for this batch
    outputs_adv = model(x_adv)
    _, predicted_adv = torch.topk(outputs_adv, k=1)
    total_attack_success += torch.sum(predicted_adv != y).item()
    total_samples += batch_size

# Compute mean number of features
mean_features_x = total_features_x / total_samples
mean_features_x_adv = total_features_x_adv / total_samples

# Compute mean attack effectiveness
mean_attack_effectiveness = (total_attack_success / total_samples) * 100

# Print results
print('Mean number of features (X):', mean_features_x)
print('Mean number of features (X_adv):', mean_features_x_adv)
print('Mean attack effectiveness:', mean_attack_effectiveness)

## Step 4 — BGA-k and BCA-k (and BCA's second version)

.clone(): This method creates a deep copy of the tensor, including its data and gradients (if any). It essentially creates a new tensor with the same data and properties as the original tensor. If the original tensor is part of a computation graph and requires gradients, the cloned tensor will also be part of the same computation graph and will require gradients. Any changes made to the cloned tensor will not affect the original tensor, and vice versa.

.copy(): This method creates a shallow copy of the tensor. It only copies the data, not the computational graph or gradients. Therefore, the copied tensor will be detached from any computation graph and will not require gradients, even if the original tensor did. Changes made to the copied tensor will not affect the original tensor, but changes in the original tensor's data will be reflected in the copied tensor.



In [ ]:
def bga_k(x, y, model, k=25, alpha=1., is_report_loss_diff=True, use_sample=False):
    """
    Multi-step bit gradient ascent
    :param x: feature vector
    :param y: ground truth labels
    :param model: neural network model
    :param k: number of steps
    :param alpha: hyperparameter for controlling updates
    :param is_report_loss_diff: flag to report loss difference
    :param use_sample: flag to sample randomly from the feasible area
    :return: the adversarial version of x according to bga_k (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), y.view(-1).long())

    # Initialize worst loss and corresponding adversarial samples
    loss_worst = loss_natural.clone()
    x_worst = x.clone()

    # Book-keeping
    sqrt_m = torch.sqrt(torch.tensor([x.size()[1]], dtype=torch.float))

    # Multi-step with gradients
    for t in range(k):
        if t == 0:
            # Initialize starting point
            x_next = get_x0(x, use_sample)
        else:
            # Compute gradient
            grad_vars = torch.autograd.grad(loss.mean(), x_var)
            grad_data = grad_vars[0].data

            # Compute the updates
            # torch.norm(grad_data, 2, 1), 2:the L2-norm , 1:the norm along dimension 1
            x_update = (sqrt_m * (1. - 2. * x_next) * grad_data >= (alpha * torch.norm(grad_data, 2, 1).unsqueeze(1))).float()

            # Find the next sample with projection to the feasible set
            x_next = xor_float_tensors(x_update, x_next)
            x_next = or_float_tensors(x_next, x)

        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, y.view(-1).long())

        # Update worst loss and adversarial samples
        replace_flag = (loss.data > loss_worst)
        loss_worst[replace_flag] = loss.data[replace_flag]
        x_worst[replace_flag] = x_next[replace_flag]

    if is_report_loss_diff:
        #print(f"Natural loss: {loss_natural.mean():.4f}, Adversarial loss: {loss_worst.mean():.4f}, Difference: {(loss_worst.mean() - loss_natural.mean()):.4f}")
        outputs = model(x_worst)
        _, predicted = torch.topk(outputs, k=1)
        done = (predicted != y).squeeze()
        print(f"bga_k: attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")

    return x_worst

In [ ]:
x_adv = bga_k(X, y, model, k=25, is_report_loss_diff=True, use_sample=False)
print(X.sum()/batch_size)
print(x_adv.sum()/batch_size)

In [ ]:
for alpha in range(1,6):
  print('alpha = ',alpha)
  x_adv = bga_k(X, y, model, k=25, alpha=alpha, is_report_loss_diff=True, use_sample=False)
  print(X.sum()/batch_size)
  print(x_adv.sum()/batch_size)
  print('***************************')

In [ ]:
def bca_k(x, y, model, k=25, is_report_loss_diff=True, use_sample=False):
    """
    Multi-step bit coordinate ascent
    :param use_sample:
    :param is_report_loss_diff:
    :param y:
    :param x: (tensor) feature vector
    :param model: nn model
    :param k: num of steps
    :return: the adversarial version of x according to bca_k (tensor)
    """

    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), y.view(-1).long())

    # keeping worst loss
    loss_worst = loss_natural.clone()
    x_worst = x.clone()

    # multi-step with gradients
    loss = None
    x_var = None
    x_next = None
    for t in range(k):
        if t == 0:
            # initialize starting point
            x_next = get_x0(x, use_sample)
        else:
            # compute gradient
            grad_vars = torch.autograd.grad(loss.mean(), x_var)
            grad_data = grad_vars[0].data

            # compute the updates (can be made more efficient than this)
            #aug_grad = (1. - 2. * x_next) * grad_data #this line is wrong because the grad_data can be negative
            aug_grad = (x_next < 0.5) * grad_data # the correct version
            val, _ = torch.topk(aug_grad, 1)
            x_update = (aug_grad >= val.expand_as(aug_grad)).float()
            # find the next sample with projection to the feasible set
            x_next = xor_float_tensors(x_update, x_next)
            x_next = or_float_tensors(x_next, x)

        # forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, y.view(-1).long())

        # update worst loss and adversarial samples
        replace_flag = (loss.data > loss_worst)
        loss_worst[replace_flag] = loss.data[replace_flag]
        x_worst[replace_flag] = x_next[replace_flag]

    if is_report_loss_diff:
        #print("Natural loss (%.4f) vs Adversarial loss (%.4f), Difference: (%.4f)" %(loss_natural.mean(), loss_worst.mean(), loss_worst.mean() - loss_natural.mean()))
        outputs = model(x_worst)
        _, predicted = torch.topk(outputs, k=1)
        done = (predicted != y).squeeze()
        print(f"bca_k: attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")



    return x_worst

In [ ]:
x_adv = bca_k(X, y, model, k=3, is_report_loss_diff=True, use_sample=False)
print(X.sum()/batch_size)
print(x_adv.sum()/batch_size)

In [ ]:
for k in range(5, 55, 5):
  print('k = ',k)
  x_adv = bca_k(X, y, model, k=k, is_report_loss_diff=True, use_sample=False)
  print(X.sum()/batch_size)
  print(x_adv.sum()/batch_size)
  print('***************************')

In [ ]:
a = torch.rand((10,10000))
a

In [ ]:
val, index = torch.topk(a, 10)

In [ ]:
val[:,-1:]

In [ ]:
def bca_k2(x, y, model, k=25, alpha=1, is_report_loss_diff=True, use_sample=False):
    """
    Multi-step bit coordinate ascent
    :param use_sample:
    :param is_report_loss_diff:
    :param y:
    :param x: (tensor) feature vector
    :param model: nn model
    :param k: num of steps
    :param alpha: top k gradients
    :return: the adversarial version of x according to bca_k (tensor)
    """

    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), y.view(-1).long())

    # keeping worst loss
    loss_worst = loss_natural.clone()
    x_worst = x.clone()

    # multi-step with gradients
    loss = None
    x_var = None
    x_next = None
    for t in range(k):
        if t == 0:
            # initialize starting point
            x_next = get_x0(x, use_sample)
        else:
            # compute gradient
            grad_vars = torch.autograd.grad(loss.mean(), x_var)
            grad_data = grad_vars[0].data

            # compute the updates (can be made more efficient than this)
            #aug_grad = (1. - 2. * x_next) * grad_data #this line is wrong because the grad_data can be negative
            aug_grad = (x_next < 0.5) * grad_data # the correct version
            val, _ = torch.topk(aug_grad, alpha)
            x_update = (aug_grad >= val[:,-1:].expand_as(aug_grad)).float()

            # find the next sample with projection to the feasible set
            x_next = xor_float_tensors(x_update, x_next)
            x_next = or_float_tensors(x_next, x)

        # forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, y.view(-1).long())

        # update worst loss and adversarial samples
        replace_flag = (loss.data > loss_worst)
        loss_worst[replace_flag] = loss.data[replace_flag]
        x_worst[replace_flag] = x_next[replace_flag]

    if is_report_loss_diff:
        #print("Natural loss (%.4f) vs Adversarial loss (%.4f), Difference: (%.4f)" %(loss_natural.mean(), loss_worst.mean(), loss_worst.mean() - loss_natural.mean()))
        outputs = model(x_worst)
        _, predicted = torch.topk(outputs, k=1)
        done = (predicted != y).squeeze()
        print(f"bca_k: attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")



    return x_worst

In [ ]:
for k in range(2, 9):
  print('k = ',k)
  x_adv = bca_k2(X, y, model, k=k, alpha=5, is_report_loss_diff=True, use_sample=False)
  print(X.sum()/batch_size)
  print(x_adv.sum()/batch_size)
  print('***************************')

In [ ]:
for i in range(1, 12):
  print('top alpha = ',i)
  x_adv = bca_k2(X, y, model, k=2, alpha=i, is_report_loss_diff=True, use_sample=False)
  print(X.sum()/batch_size)
  print(x_adv.sum()/batch_size)
  print('***************************')

## Step 5 — Grosse-k, in both directions

In [ ]:
def grosse_k(x, y, model, k=25, is_report_loss_diff=True, use_sample=False):
    """
    Multi-step bit coordinate ascent
    :param use_sample:
    :param is_report_loss_diff:
    :param y:
    :param x: (tensor) feature vector
    :param model: nn model
    :param k: num of steps
    :return: the adversarial version of x according to bca_k (tensor)
    """
    epsilon = 1e-10 # avoid gradient less than epsilon

    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), y.view(-1).long())

    # keeping worst loss
    loss_worst = loss_natural.clone()
    x_worst = x.clone()

    # multi-step with gradients
    output = None
    x_var = None
    x_next = None
    for t in range(k):
        if t == 0:
            # initialize starting point
            x_next = get_x0(x, use_sample)
        else:
            # compute gradient
            # ouput.shape=([batch_size, 2]) because of 2 neoruns, so we just use the output of the first neorun(benign)
            grad_vars = torch.autograd.grad(output[:, 0].mean(), x_var)
            grad_data = grad_vars[0].data

            # compute the updates (can be made more efficient than this)
            #aug_grad = (1. - x_next) * grad_data
            aug_grad = (x_next < 0.5) * grad_data
            val, _ = torch.topk(aug_grad, 1)
            x_update = ((aug_grad >= val.expand_as(aug_grad)).float()) * (aug_grad > epsilon)

            # find the next sample with projection to the feasible set
            x_next = xor_float_tensors(x_update, x_next)
            x_next = or_float_tensors(x_next, x)

        # forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        output = model(x_var)
        loss = criterion(output, y.view(-1).long())

        # update worst loss and adversarial samples
        replace_flag = (loss.data > loss_worst)
        loss_worst[replace_flag] = loss.data[replace_flag]
        x_worst[replace_flag] = x_next[replace_flag]

    if is_report_loss_diff:
        #print("Natural loss (%.4f) vs Adversarial loss (%.4f), Difference: (%.4f)" %(loss_natural.mean(), loss_worst.mean(), loss_worst.mean() - loss_natural.mean()))
        outputs = model(x_worst)
        _, predicted = torch.topk(outputs, k=1)
        done = (predicted != y).squeeze()
        print(f"grosse_k: attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")



    return x_worst

In [ ]:
# adversarial examples for malwares(to show them as benign)
x_adv = grosse_k(X[y.squeeze()==1], y[y.squeeze()==1], model, k=15, is_report_loss_diff=True, use_sample=False)
print(X[y.squeeze()==1].sum()/len(X[y.squeeze()==1]))
print(x_adv.sum()/len(x_adv))

In [ ]:
# adversarial examples for benigns(to show them as malware)
# does not work, because its based on first neron which is for converting malware into benign
a = grosse_k(X[y.squeeze()==0], y[y.squeeze()==0], model, k=25, is_report_loss_diff=True, use_sample=False)

## Step 6 — PGD

In [ ]:
def pgd(x, y, model, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack.
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), y.view(-1).long())

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)

    # Multi-step PGD
    for t in range(k):
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, y.view(-1).long())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        grad_data = grad_vars[0].data
        gradients = grad_data * (x < 0.5)

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients)
        elif norm == 'l2_2':
            max_grad, _ = gradients.max(dim=1, keepdim=True)
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            #print(max_grad/l2norm)
            perturbation = torch.minimum(torch.tensor(1., dtype=x.dtype, device=x.device), gradients / max_grad)

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = torch.minimum(torch.tensor(1., dtype=x.dtype, device=x.device), gradients / l2norm)
            perturbation[torch.isnan(perturbation)] = 0.
            perturbation[torch.isinf(perturbation)] = 1.
        elif norm == 'l1':
            #ignore the gradient of indice which is updated
            gradients = gradients * (x_next < 0.5)
            val, _ = torch.topk(gradients, 1)
            perturbation = (gradients >= val.expand_as(gradients)).float()
            # stop perturbing the examples that are successful to evade the victim
            outputs = model(x_next)
            _, predicted = torch.topk(outputs, k=1)
            done = (predicted != y).squeeze()

            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Feasible projection
    x_next = or_float_tensors(x_next, x)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), y.view(-1).long()).data

    if is_report_loss_diff:
        outputs = model(x_next)
        _, predicted = torch.topk(outputs, k=1)
        done = (predicted != y).squeeze()
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv < loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    return x_next

In [ ]:
x_adv = pgd(X[y.squeeze()==1],  y[y.squeeze()==1], model, k=25, step_length=1., norm='l1', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)
print(X[y.squeeze()==1].sum()/len(X[y.squeeze()==1]))
print(x_adv.sum()/len(x_adv))

In [ ]:
x_adv = bca_k(X[y.squeeze()==1], y[y.squeeze()==1], model, k=25, is_report_loss_diff=True, use_sample=False)
print(X[y.squeeze()==1].sum()/len(X[y.squeeze()==1]))
print(x_adv.sum()/len(x_adv))

In [ ]:
x_adv = pgd(X, y, model, k=250, step_length=0.002, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)
print(X.sum()/batch_size)
print(x_adv.sum()/batch_size)

In [ ]:
x_adv = pgd(X[y.squeeze()==1], y[y.squeeze()==1], model, k=250, step_length=0.003, norm='l2_2', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)
print(X[y.squeeze()==1].sum()/len(X[y.squeeze()==1]))
print(x_adv.sum()/len(x_adv))

In [ ]:
x_adv = pgd(X[y.squeeze()==1], y[y.squeeze()==1], model, k=250, step_length=0.05, norm='l2', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)
print(X[y.squeeze()==1].sum()/len(X[y.squeeze()==1]))
print(x_adv.sum()/len(x_adv))

## Step 7 — A metrics harness for all of the above

In [ ]:
def compute_attack_metrics(test_loader, model, attack_type, k, step_length, norm, initial_rounding_threshold, round_threshold):
    total_samples = 0
    total_attack_success = 0
    total_features_x = 0
    total_features_x_adv = 0

    for X, y in test_loader:
        X = X[y.squeeze()==1]
        y = y[y.squeeze()==1]
        batch_size = X.size(0)
        x_adv = pgd(X, y, model, k=k, step_length=step_length, norm=norm, initial_rounding_threshold=initial_rounding_threshold, round_threshold=round_threshold, random=False, is_report_loss_diff=False, is_sample=False)

        # Compute mean number of features
        total_features_x += X.sum().item()
        total_features_x_adv += x_adv.sum().item()

        # Compute attack effectiveness for this batch
        outputs_adv = model(x_adv)
        _, predicted_adv = torch.topk(outputs_adv, k=1)
        total_attack_success += torch.sum(predicted_adv != y).item()
        total_samples += batch_size

    # Compute mean number of features
    mean_features_x = total_features_x / total_samples
    mean_features_x_adv = total_features_x_adv / total_samples

    # Compute mean attack effectiveness
    mean_attack_effectiveness = (total_attack_success / total_samples) * 100

    return mean_features_x, mean_features_x_adv, mean_attack_effectiveness

# for l2_2
mean_features_x_l2_2, mean_features_x_adv_l2_2, mean_attack_effectiveness_l2_2 = compute_attack_metrics(test_loader, model, 'l2_2', k=250, step_length=0.003, norm='l2_2', initial_rounding_threshold=0.5, round_threshold=0.5)

# Print results for l2_2 attack
print('For l2_2 attack:')
print('Mean number of features (X):', mean_features_x_l2_2)
print('Mean number of features (X_adv):', mean_features_x_adv_l2_2)
print('Mean attack effectiveness:', mean_attack_effectiveness_l2_2)



# for l2
mean_features_x_l2, mean_features_x_adv_l2, mean_attack_effectiveness_l2 = compute_attack_metrics(test_loader, model, 'l2', k=250, step_length=0.05, norm='l2', initial_rounding_threshold=0.5, round_threshold=0.5)

# Print results for l2 attack
print('For l2 attack:')
print('Mean number of features (X):', mean_features_x_l2)
print('Mean number of features (X_adv):', mean_features_x_adv_l2)
print('Mean attack effectiveness:', mean_attack_effectiveness_l2)

In [ ]:
x = X
steps_max=5
attack_list = ['linf', 'l2', 'l1']
is_sample = False
varepsilon = 1e-20

model.eval()
with torch.no_grad():
    loss, done = get_loss(x,y,model) #shape:[samples],[samples]
pre_loss = loss
n, red_n = x.shape[0], x.shape[1]
adv_x = x.detach().clone()
stop_flag = torch.zeros(n, dtype=torch.bool) #[samples]

for t in range(steps_max):
  num_sample_red = n - torch.sum(stop_flag)
  print('num_sample_red : ',num_sample_red)
  if num_sample_red <= 0:
      break

  red_label = y[~stop_flag]
  pertbx = []


  for norm in attack_list:
      if norm=='l1':
        l1 = pgd(adv_x[~stop_flag], red_label, model, k=50, step_length=1., norm='l1', is_sample=is_sample)
        pertbx.append(l1)
      elif norm=='l2':
        l2 = pgd(adv_x[~stop_flag], red_label, model, k=200, step_length=0.05, norm='l2', is_sample=is_sample)
        pertbx.append(l2)
      elif norm=='linf':
        linf = pgd(adv_x[~stop_flag], red_label, model, k=500, step_length=0.002, norm='linf', is_sample=is_sample)
        pertbx.append(linf)
      else :
        raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

  # here pertbx.shape = a list of (number of attacks  ,(num_sample_red,features))
  pertbx = torch.vstack(pertbx)
  # here pertbx.shape = a tensor (num_sample_red*number of attacks samples, features)

  with torch.no_grad():
    red_label_ext = torch.cat([red_label] * len(attack_list)) #(labels*number of attacks )
    loss, done = get_loss(pertbx, red_label_ext,model) #(labels*number of attacks )
    loss = loss.reshape(len(attack_list), num_sample_red).permute(1, 0) #(num_sample_red,number of attacks)
    done = done.reshape(len(attack_list), num_sample_red).permute(1, 0) #(num_sample_red,number of attacks)

    success_flag = torch.any(done, dim=-1) #(num_sample_red)
    # for a sample, if there is at least one successful attack, we will select the one with maximum loss;
    # while if no attacks evade the victim successful, all perturbed examples are reminded for selection

    done[~torch.any(done, dim=-1)] = 1 #loss.shape=done.shape=(samples,number of attacks)
    loss = (loss * done.to(torch.float)) + torch.min(loss) * (~done).to(torch.float) #(num_sample_red,number of attacks)
    pertbx = pertbx.reshape(len(attack_list), num_sample_red, red_n).permute([1, 0, 2])#(num_sample_red,attacks,features)
    _, indices = loss.max(dim=-1) # ans:(samples), max loss among attacks which worked, and max loss among all attacks for sample , none of them worked
    adv_x[~stop_flag] = pertbx[torch.arange(num_sample_red), indices]
    a_loss = loss[torch.arange(num_sample_red), indices]
    pre_stop_flag = stop_flag.clone()
    stop_flag[~stop_flag] = (torch.abs(pre_loss[~stop_flag] - a_loss) < varepsilon) | success_flag
    pre_loss[~pre_stop_flag] = a_loss

## Step 8 — PGD-Max: run every norm, keep the best

In [ ]:
def PGD_Max(x,y, model, attack_list = ['linf', 'l2', 'l1'],steps_max=5, is_sample = False, varepsilon = 1e-20):
    """
    PGD_Max adversarial attack.

    Args:
        x: Input data tensor (shape: [samples, features])
        y: Ground truth labels tensor (shape: [samples])
        model: Neural network model
        attack_list: List of norms for attacks (default: ['linf', 'l2', 'l1'])
        steps_max: Maximum number of steps (default: 5)
        is_sample: Flag to sample randomly from the feasible area (default: False)
        vaρεpsilon: Tolerance for stopping condition (default: 1e-20)

    Returns:
        Adversarial version of input data (tensor)
    """
    batch_size = x.shape[0]
    norm_params = {
        'l1': {'k': 50, 'step_length': 1.0},
        'l2': {'k': 200, 'step_length': 0.05},
        'linf': {'k': 500, 'step_length': 0.002}
    }

    model.eval()
    with torch.no_grad():
        loss, done = get_loss(x,y,model) #shape:[samples],[samples]

    pre_loss = loss
    n = x.shape[0]
    adv_x = x.detach().clone()
    stop_flag = torch.zeros(n, dtype=torch.bool) #[samples]

    for t in range(steps_max):
      num_remaining  = (~stop_flag).sum().item()
      print('number of remaining samples : ',num_remaining )
      if num_remaining  <= 0:
          break

      remaining_label = y[~stop_flag]
      pertbx = []

      for norm in attack_list:
          if norm in norm_params:
              params = norm_params[norm]
              perturbation = pgd(adv_x[~stop_flag], remaining_label, model, norm=norm, is_sample=is_sample, **params)
              print("the number of added features : ", (perturbation.sum() - adv_x[~stop_flag].sum())/len(adv_x[~stop_flag]))
              pertbx.append(perturbation)
          else:
              raise ValueError("Expected 'l1' or 'l2' or 'linf' norm.")


      # here pertbx.shape = a list of (number of attacks  ,(num_remaining ,features))
      pertbx = torch.vstack(pertbx)
      # here pertbx.shape = a tensor (num_remaining *number of attacks samples, features)

      with torch.no_grad():
        remaining_label_ext = torch.cat([remaining_label] * len(attack_list)) #(labels*number of attacks )
        loss, done = get_loss(pertbx, remaining_label_ext,model) #(labels*number of attacks )
        loss = loss.reshape(len(attack_list), num_remaining ).permute(1, 0) #(num_remaining ,number of attacks)
        done = done.reshape(len(attack_list), num_remaining ).permute(1, 0) #(num_remaining ,number of attacks)

        success_flag = torch.any(done, dim=-1) #(num_remaining )
        # for a sample, if there is at least one successful attack, we will select the one with maximum loss;
        # while if no attacks evade the victim successful, all perturbed examples are reminded for selection

        done[~torch.any(done, dim=-1)] = 1 #loss.shape=done.shape=(samples,number of attacks)
        loss = (loss * done.to(torch.float)) + torch.min(loss) * (~done).to(torch.float) #(num_remaining ,number of attacks)
        pertbx = pertbx.reshape(len(attack_list), num_remaining , x.shape[1]).permute([1, 0, 2])#(num_remaining ,attacks,features)
        _, indices = loss.max(dim=-1) # ans:(samples), max loss among attacks which worked, and max loss among all attacks for sample , none of them worked
        adv_x[~stop_flag] = pertbx[torch.arange(num_remaining ), indices]
        a_loss = loss[torch.arange(num_remaining ), indices]
        pre_stop_flag = stop_flag.clone()
        stop_flag[~stop_flag] = (torch.abs(pre_loss[~stop_flag] - a_loss) < varepsilon) | success_flag
        pre_loss[~pre_stop_flag] = a_loss

    return adv_x

In [ ]:
def PGD_Max2(x,y, model, attack_list = ['linf', 'l2', 'l1'],steps_max=5, is_sample = False, varepsilon = 1e-20):
    """
    PGD_Max adversarial attack.

    Args:
        x: Input data tensor (shape: [samples, features])
        y: Ground truth labels tensor (shape: [samples])
        model: Neural network model
        attack_list: List of norms for attacks (default: ['linf', 'l2', 'l1'])
        steps_max: Maximum number of steps (default: 5)
        is_sample: Flag to sample randomly from the feasible area (default: False)
        vaρεpsilon: Tolerance for stopping condition (default: 1e-20)

    Returns:
        Adversarial version of input data (tensor)
    """
    batch_size = x.shape[0]
    norm_params = {
        'l1': {'k': 50, 'step_length': 1.0},
        'l2': {'k': 200, 'step_length': 0.05},
        'linf': {'k': 500, 'step_length': 0.002}
    }

    model.eval()
    with torch.no_grad():
        loss, done = get_loss(x,y,model) #shape:[samples],[samples]

    pre_loss = loss
    n = x.shape[0]
    adv_x = x.detach().clone()
    stop_flag = torch.zeros(n, dtype=torch.bool) #[samples]

    for t in range(steps_max):
      num_remaining  = (~stop_flag).sum().item()
      print('number of remaining samples : ',num_remaining )
      if num_remaining  <= 0:
          break

      remaining_label = y[~stop_flag]
      pertbx = []

      for norm in attack_list:
          if norm in norm_params:
              params = norm_params[norm]
              perturbation = pgd(adv_x[~stop_flag], remaining_label, model, norm=norm, is_sample=is_sample, **params)
              print("the number of added features : ", (perturbation.sum() - adv_x[~stop_flag].sum())/len(adv_x[~stop_flag]))
              pertbx.append(perturbation)
          else:
              raise ValueError("Expected 'l1' or 'l2' or 'linf' norm.")


      # here pertbx.shape = a list of (number of attacks  ,(num_remaining ,features))
      pertbx = torch.vstack(pertbx)
      # here pertbx.shape = a tensor (num_remaining *number of attacks samples, features)

      with torch.no_grad():
        remaining_label_ext = torch.cat([remaining_label] * len(attack_list)) #(labels*number of attacks )
        loss, done = get_loss(pertbx, remaining_label_ext,model) #(labels*number of attacks )

        # for a sample, if there is at least one successful attack, we will select the one with maximum loss;
        # while if no attacks evade the victim successful, all perturbed examples are reminded for selection
        max_v = loss.amax()
        loss[done] += max_v

        loss = loss.reshape(len(attack_list), num_remaining ).permute(1, 0) #(num_remaining ,number of attacks)
        done = done.reshape(len(attack_list), num_remaining ).permute(1, 0) #(num_remaining ,number of attacks)

        success_flag = torch.any(done, dim=-1) #(num_remaining )

        pertbx = pertbx.reshape(len(attack_list), num_remaining , x.shape[1]).permute([1, 0, 2])#(num_remaining ,attacks,features)
        _, indices = loss.max(dim=-1) # ans:(samples), max loss among attacks which worked, and max loss among all attacks for sample , none of them worked
        adv_x[~stop_flag] = pertbx[torch.arange(num_remaining ), indices]
        a_loss = loss[torch.arange(num_remaining ), indices]
        pre_stop_flag = stop_flag.clone()
        stop_flag[~stop_flag] = (torch.abs(pre_loss[~stop_flag] - a_loss) < varepsilon) | success_flag
        pre_loss[~pre_stop_flag] = a_loss

    return adv_x

In [ ]:
adv_x = PGD_Max2(X,y, model)
print(X.sum()/batch_size)
print(adv_x.sum()/batch_size)

In [ ]:
adv_x = PGD_Max2(X[y.squeeze()==1],y[y.squeeze()==1], model)
print(X[y.squeeze()==1].sum()/len(X[y.squeeze()==1]))
print(adv_x.sum()/len(X[y.squeeze()==1]))

## Step 9 — StepwiseMax and the one-step variants

In [ ]:
def pgd_step(x, y, model, norm, k, step_length):
    """
    Projected Gradient Descent (PGD) adversarial attack for stepwise.
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :return: The adversarial version of x (tensor)(not rounded)
    """
    model.eval()

    criterion = nn.CrossEntropyLoss(reduction='none')

    # Initialize starting point
    x_next = x.clone()

    # Multi-step PGD
    for t in range(k):
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, y.view(-1).long())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        grad_data = grad_vars[0].data
        gradients = grad_data * (x < 0.5)

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients)
        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = torch.minimum(torch.tensor(1., dtype=x.dtype, device=x.device), gradients / l2norm)
            perturbation[torch.isnan(perturbation)] = 0.
            perturbation[torch.isinf(perturbation)] = 1.
        elif norm == 'l1':
            #ignore the gradient of indice which is updated
            gradients = gradients * (x_next < 0.5)
            val, _ = torch.topk(gradients, 1)
            perturbation = torch.sign(gradients >= val.expand_as(gradients))
            # stop perturbing the examples that are successful to evade the victim
            outputs = model(x_next)
            _, predicted = torch.topk(outputs, k=1)
            done = (predicted != y).squeeze()
            if torch.all(done):
                break
            perturbation[done] = 0.
        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    #remove negative pertubations, we cant use OR function because we have values between (0,1) like 0.2 which we want to keep
    x_adv = (((x_next - x) >= 0) * x_next) + (((x_next - x) < 0) * x)
    return x_adv

In [ ]:
step_checks = 1
steps = 100
is_attacker = False
is_score_round = False
use_random = False
round_threshold = 0.5
x = X
label = y
attack_list = ['linf', 'l2', 'l1']
norm_params = {
    'l1': {'step_length': 1.0},
    'l2': {'step_length': 0.05},
    'linf': {'step_length': 0.002}
}

model.eval()
if not is_attacker:
    step_checks = [1, 10, 25, 50]
    step_check = random.choice(step_checks)
mini_steps = [step_check] * (steps // step_check)
mini_steps = mini_steps + [steps % step_check] if steps % step_check != 0 else mini_steps

print('mini steps = ', mini_steps)

n, red_n = x.size()[0], x.size()[1]
adv_x = x.detach().clone()
pert_x_cont = None
prev_done = None
for i, mini_step in enumerate(mini_steps):
    with torch.no_grad():
        if i == 0 :
            adv_x = get_x0(adv_x, initial_rounding_threshold=round_threshold, is_sample=use_random)

        _, done = get_loss(adv_x, label, model)
    if torch.all(done):
        break
    if i == 0:
        adv_x[~done] = x[~done]  # recompute the perturbation under other penalty factors
        prev_done = done
    else:
        adv_x[~done] = pert_x_cont[~done[~prev_done]]
        prev_done = done

    num_sample_red = torch.sum(~done).item()
    pertbx = []
    for norm in attack_list:
        params = norm_params[norm]
        perturbation = pgd_step(adv_x[~done], label[~done], model, norm=norm, k=mini_step, **params)
        print("the number of added features(not rounded) ", norm,": ", perturbation.sum()/len(adv_x[~done]) - adv_x[~done].sum()/len(adv_x[~done]))
        pertbx.append(perturbation)
    with torch.no_grad():
        n_attacks = len(pertbx)
        pertbx = torch.vstack(pertbx)
        label_ext = torch.cat([label[~done]] * n_attacks)
        if (not is_attacker) and (not is_score_round):
            scores, _done = get_loss(pertbx, label_ext,model)
        else:
            scores, _done = get_loss(round_x(pertbx, round_threshold), label_ext, model)
        max_v = scores.amax() if scores.amax() > 0 else 0.
        scores[_done] += max_v
        pertbx = pertbx.reshape(n_attacks, num_sample_red, red_n).permute([1, 0, 2])
        scores = scores.reshape(n_attacks, num_sample_red).permute(1, 0)
        _2, s_idx = scores.max(dim=-1)
        pert_x_cont = pertbx[torch.arange(num_sample_red), s_idx]
        adv_x[~done] = pert_x_cont if not is_attacker else round_x(pert_x_cont, round_threshold)


if is_attacker:
    adv_x = round_x(adv_x, round_threshold)
with torch.no_grad():
    _, done = get_loss(adv_x, label, model)
    print(f"step-wise max: attack effectiveness {done.sum().item() / done.size()[0] * 100:.3f}%.")

In [ ]:
def StepwiseMax(
    x,
    label,
    model,
    attack_list=["linf", "l2", "l1"],
    step_lengths={"l1": 1.0, "l2": 0.05, "linf": 0.002},
    steps=100,
    step_check = 1,
    random_start=False,
    round_threshold=0.5,
    is_attacker=False,
    is_score_round = False
):
  """
    Stepwise max attack (mixture of pgd-l1, pgd-l2, pgd-linf).

    Args:
        x: Input data tensor (shape: [batch_size, feature_dim])
        label: Ground truth labels tensor (shape: [batch_size])
        model: Victim model
        attack_list: List of attack norms (default: ["linf", "l2", "l1"])
        step_lengths: Dictionary mapping norm to its step length (default: {"l1": 1.0, "l2": 0.05, "linf": 0.002})
        steps: Maximum number of iterations (default: 100)
        random_start: Use random starting point (default: False)
        round_threshold: Threshold for rounding real scalars (default: 0.5)
        is_attacker: Play the role of attacker (default: False)

    Returns:
        Adversarial examples tensor (same shape as x)
  """

  model.eval()

  step_check = 1
  if not is_attacker:
      step_checks = [1, 10, 25, 50]
      step_check = random.choice(step_checks)

  print(f"Step check: {step_check}")
  mini_steps = [step_check] * (steps // step_check)
  if steps % step_check != 0:
      mini_steps.append(steps % step_check)

  n, red_n = x.shape
  adv_x = x.detach().clone()
  pert_x_cont = None
  prev_done = None
  for i, mini_step in enumerate(mini_steps):
      with torch.no_grad():
          if i == 0 :
              adv_x = get_x0(adv_x, initial_rounding_threshold=round_threshold, is_sample=random_start)

          _, done = get_loss(adv_x, label, model)
      if torch.all(done):
          break
      if i == 0:
          adv_x[~done] = x[~done]  # recompute the perturbation under other penalty factors
          prev_done = done
      else:
          adv_x[~done] = pert_x_cont[~done[~prev_done]]
          prev_done = done

      num_sample_red = torch.sum(~done).item()
      pertbx = []
      for norm in attack_list:
          step_length = step_lengths.get(norm, step_lengths["l1"])
          perturbation = pgd_step(adv_x[~done], label[~done], model, norm, mini_step, step_length)
          #print("the number of added features(not rounded) ", norm,": ", perturbation.sum()/len(adv_x[~done]) - adv_x[~done].sum()/len(adv_x[~done]))
          #print("the number of added features(rounded) ", norm," : ",(round_x(perturbation, round_threshold).sum() - round_x(adv_x[~done], round_threshold).sum())/len(adv_x[~done]))
          pertbx.append(perturbation)
      with torch.no_grad():
          pertbx = torch.vstack(pertbx)

          n_attacks = len(attack_list)
          label_ext = torch.cat([label[~done]] * n_attacks)

          if (not is_attacker) and (not is_score_round):
              scores, _done = get_loss(pertbx, label_ext,model)
          else:
              scores, _done = get_loss(round_x(pertbx, round_threshold), label_ext, model)
          max_v = scores.amax() if scores.amax() > 0 else 0.
          scores[_done] += max_v

          pertbx = pertbx.reshape(n_attacks, num_sample_red, red_n).permute([1, 0, 2])
          scores = scores.reshape(n_attacks, num_sample_red).permute(1, 0)
          _2, s_idx = scores.max(dim=-1)
          pert_x_cont = pertbx[torch.arange(num_sample_red), s_idx]
          adv_x[~done] = pert_x_cont if not is_attacker else round_x(pert_x_cont, round_threshold)

  print(i)
  if is_attacker:
      adv_x = round_x(adv_x, round_threshold)
  with torch.no_grad():
      _, done = get_loss(adv_x, label, model)
      print(f"step-wise max: attack effectiveness {done.sum().item() / done.size()[0] * 100:.3f}%.")

  return adv_x

In [ ]:
adv_x = StepwiseMax(X, y, model)
print(X.sum()/batch_size)
print(adv_x.sum()/batch_size)

In [ ]:
adv_x = StepwiseMax(X, y, model,is_attacker=True)
print(X.sum()/batch_size)
print(adv_x.sum()/batch_size)

In [ ]:
def StepwiseMax2(
    x,
    label,
    model,
    attack_list=["linf", "l2", "l1"],
    step_lengths={"l1": 1.0, "l2": 0.05, "linf": 0.002},
    steps=100,
    step_check = 1,
    random_start=False,
    round_threshold=0.5,
    is_attacker=False,
    is_score_round = False
):
  """
    Stepwise max attack (mixture of pgd-l1, pgd-l2, pgd-linf).

    Args:
        x: Input data tensor (shape: [batch_size, feature_dim])
        label: Ground truth labels tensor (shape: [batch_size])
        model: Victim model
        attack_list: List of attack norms (default: ["linf", "l2", "l1"])
        step_lengths: Dictionary mapping norm to its step length (default: {"l1": 1.0, "l2": 0.05, "linf": 0.002})
        steps: Maximum number of iterations (default: 100)
        random_start: Use random starting point (default: False)
        round_threshold: Threshold for rounding real scalars (default: 0.5)
        is_attacker: Play the role of attacker (default: False)

    Returns:
        Adversarial examples tensor (same shape as x)
  """

  model.eval()

  step_check = 1
  if not is_attacker:
      step_checks = [1, 10, 25, 50]
      step_check = random.choice(step_checks)

  print(f"Step check: {step_check}")
  mini_steps = [step_check] * (steps // step_check)
  if steps % step_check != 0:
      mini_steps.append(steps % step_check)
  n, red_n = x.shape
  adv_x = x.detach().clone()
  pert_x_cont = None
  prev_done = None
  for i, mini_step in enumerate(mini_steps):
      with torch.no_grad():
          if i == 0 :
              adv_x = get_x0(adv_x, initial_rounding_threshold=round_threshold, is_sample=random_start)

          if is_attacker:
            _, done = get_loss(round_x(adv_x, round_threshold), label, model)
          else :
            _, done = get_loss(adv_x, label, model)
      if torch.all(done):
          break
      if i == 0:
          adv_x[~done] = x[~done]  # recompute the perturbation under other penalty factors
          prev_done = done
      else:
          adv_x[~done] = pert_x_cont[~done[~prev_done]]
          prev_done = done

      print('len(adv_x[~done]) : ',len(adv_x[~done]))
      num_sample_red = torch.sum(~done).item()
      pertbx = []
      for norm in attack_list:
          step_length = step_lengths.get(norm, step_lengths["l1"])
          perturbation = pgd_step(adv_x[~done], label[~done], model, norm, mini_step, step_length)
          #print("the number of added features(not rounded) ", norm,": ", perturbation.sum()/len(adv_x[~done]) - adv_x[~done].sum()/len(adv_x[~done]))
          #print("the number of added features(rounded) ", norm," : ",(round_x(perturbation, round_threshold).sum() - round_x(adv_x[~done], round_threshold).sum())/len(adv_x[~done]))
          pertbx.append(perturbation)
      with torch.no_grad():
          pertbx = torch.vstack(pertbx)

          n_attacks = len(attack_list)
          label_ext = torch.cat([label[~done]] * n_attacks)

          if (is_score_round):
              scores, _done = get_loss(round_x(pertbx, round_threshold), label_ext, model)
          elif is_attacker:
              scores, _ = get_loss(pertbx, label_ext,model)
              _, _done = get_loss(round_x(pertbx, round_threshold), label_ext, model)
          else:
              scores, _done = get_loss(pertbx, label_ext,model)

          max_v = scores.amax() if scores.amax() > 0 else 0.
          scores[_done] += max_v

          pertbx = pertbx.reshape(n_attacks, num_sample_red, red_n).permute([1, 0, 2])
          scores = scores.reshape(n_attacks, num_sample_red).permute(1, 0)
          _2, s_idx = scores.max(dim=-1)
          pert_x_cont = pertbx[torch.arange(num_sample_red), s_idx]
          adv_x[~done] = pert_x_cont

  print(i)
  if is_attacker:
      adv_x = round_x(adv_x, round_threshold)
  with torch.no_grad():
      _, done = get_loss(adv_x, label, model)
      print(f"step-wise max: attack effectiveness {done.sum().item() / done.size()[0] * 100:.3f}%.")

  return adv_x

In [ ]:
def pgd_one_step(x, y, model, step_lengths):
    """
    Projected Gradient Descent (PGD) adversarial attack for stepwise.
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :return: The adversarial version of x (tensor)(not rounded)
    """
    model.eval()

    criterion = nn.CrossEntropyLoss(reduction='none')

    # Initialize starting point
    x_next = x.clone()

    # one-step PGD

    # Forward pass
    x_var = x_next.clone().detach().requires_grad_(True)
    y_model = model(x_var)
    loss = criterion(y_model, y.view(-1).long())

    # Compute gradient
    grad_vars = torch.autograd.grad(loss.mean(), x_var)
    grad_data = grad_vars[0].data
    gradients = grad_data * (x < 0.5)


    # Norms
    pertbx = []
    # norm = linf
    step_length = step_lengths.get("linf", step_lengths["l1"])
    perturbation_linf = torch.sign(gradients)
    x_next_linf = torch.clamp(x_next + perturbation_linf * step_length, min=0., max=1.)
    #remove negative pertubations, we cant use OR function because we have values between (0,1) like 0.2 which we want to keep
    x_adv_linf = (((x_next_linf - x) >= 0) * x_next_linf) + (((x_next_linf - x) < 0) * x)
    pertbx.append(x_adv_linf)
    # norm = l2
    step_length = step_lengths.get("l2", step_lengths["l1"])
    l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
    perturbation_l2 = torch.minimum(torch.tensor(1., dtype=x.dtype, device=x.device), gradients / l2norm)
    perturbation_l2[torch.isnan(perturbation_l2)] = 0.
    perturbation_l2[torch.isinf(perturbation_l2)] = 1.
    x_next_l2 = torch.clamp(x_next + perturbation_l2 * step_length, min=0., max=1.)
    x_adv_l2 = (((x_next_l2 - x) >= 0) * x_next_l2) + (((x_next_l2 - x) < 0) * x)
    pertbx.append(x_adv_l2)
    # norm = l1
    step_length = step_lengths.get("l1", step_lengths["l1"])
    #ignore the gradient of indice which is updated
    gradients = gradients * (x_next < 0.5)
    val, _ = torch.topk(gradients, 1)
    perturbation_l1 = torch.sign(gradients >= val.expand_as(gradients))
    x_next_l1 = torch.clamp(x_next + perturbation_l1 * step_length, min=0., max=1.)
    x_adv_l1 = (((x_next_l1 - x) >= 0) * x_next_l1) + (((x_next_l1 - x) < 0) * x)
    pertbx.append(x_adv_l1)

    return pertbx

In [ ]:
def StepwiseMax_onestep(
    x,
    label,
    model,
    attack_list=["linf", "l2", "l1"],
    step_lengths={"l1": 1.0, "l2": 0.05, "linf": 0.002},
    steps=100,
    random_start=False,
    round_threshold=0.5,
):
  """
    Stepwise max attack (mixture of pgd-l1, pgd-l2, pgd-linf).

    Args:
        x: Input data tensor (shape: [batch_size, feature_dim])
        label: Ground truth labels tensor (shape: [batch_size])
        model: Victim model
        attack_list: List of attack norms (default: ["linf", "l2", "l1"])
        step_lengths: Dictionary mapping norm to its step length (default: {"l1": 1.0, "l2": 0.05, "linf": 0.002})
        steps: Maximum number of iterations (default: 100)
        random_start: Use random starting point (default: False)
        round_threshold: Threshold for rounding real scalars (default: 0.5)

    Returns:
        Adversarial examples tensor (same shape as x)
  """

  model.eval()

  n, red_n = x.shape
  adv_x = x.detach().clone()
  pert_x_cont = None
  prev_done = None
  for step in range(steps):
      with torch.no_grad():
          if step == 0 :
              adv_x = get_x0(adv_x, initial_rounding_threshold=round_threshold, is_sample=random_start)

          _, done = get_loss(round_x(adv_x, round_threshold), label, model)

      if torch.all(done):
          break
      if step == 0:
          adv_x[~done] = x[~done]  # recompute the perturbation under other penalty factors
          prev_done = done
      else:
          adv_x[~done] = pert_x_cont[~done[~prev_done]]
          prev_done = done

      print('len(adv_x[~done]) : ',len(adv_x[~done]))
      num_sample_red = torch.sum(~done).item()

      pertbx = pgd_one_step(adv_x[~done], label[~done], model, step_lengths)

      with torch.no_grad():
          pertbx = torch.vstack(pertbx)

          n_attacks = len(attack_list)
          label_ext = torch.cat([label[~done]] * n_attacks)

          scores, _ = get_loss(pertbx, label_ext,model)
          _, _done = get_loss(round_x(pertbx, round_threshold), label_ext, model)

          max_v = scores.amax() if scores.amax() > 0 else 0.
          scores[_done] += max_v

          pertbx = pertbx.reshape(n_attacks, num_sample_red, red_n).permute([1, 0, 2])
          scores = scores.reshape(n_attacks, num_sample_red).permute(1, 0)
          _2, s_idx = scores.max(dim=-1)
          pert_x_cont = pertbx[torch.arange(num_sample_red), s_idx]
          adv_x[~done] = pert_x_cont

  print(step)
  adv_x = round_x(adv_x, round_threshold)
  with torch.no_grad():
      _, done = get_loss(adv_x, label, model)
      print(f"step-wise max: attack effectiveness {done.sum().item() / done.size()[0] * 100:.3f}%.")

  return adv_x

In [ ]:
def around_x(x, std_deviation=0.1, random_start=False):
    """
    Helper function to randomly initialize the inner maximizer algorithm.
    Randomizes the input tensor while preserving its functionality.
    :param x: input tensor
    :param std_deviation: std_deviation for domain
    :param random_start: flag to sample randomly from feasible area
    :return: randomly sampled feasible version of x
    """
    if random_start:

        # Generate random tensor from a Gaussian distribution centered around zero
        random_tensor = abs(torch.randn(x.size()))

        # Scale the values to control the spread of the distribution
        random_tensor *= std_deviation

        return torch.clamp(x + random_tensor, min=0., max=1.)
    else:
        return x

In [ ]:
def pgd_one_step2(x, y, model, step_lengths,x_initial):
    """
    Projected Gradient Descent (PGD) adversarial attack for stepwise.
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :return: The adversarial version of x (tensor)(not rounded)
    """
    model.eval()

    criterion = nn.CrossEntropyLoss(reduction='none')

    # Initialize starting point
    x_next = x.clone()

    # one-step PGD

    # Forward pass
    x_var = x_next.clone().detach().requires_grad_(True)
    y_model = model(x_var)
    loss = criterion(y_model, y.view(-1).long())

    # Compute gradient
    grad_vars = torch.autograd.grad(loss.mean(), x_var)
    grad_data = grad_vars[0].data
    gradients = grad_data * (x_initial < 0.5)


    # Norms
    pertbx = []
    # norm = linf
    step_length = step_lengths.get("linf", step_lengths["l1"])
    perturbation_linf = torch.sign(gradients)
    x_next_linf = torch.clamp(x_next + perturbation_linf * step_length, min=0., max=1.)
    #remove negative pertubations, we cant use OR function because we have values between (0,1) like 0.2 which we want to keep
    x_adv_linf = (((x_next_linf - x_initial) >= 0) * x_next_linf) + (((x_next_linf - x_initial) < 0) * x_initial)
    pertbx.append(x_adv_linf)
    # norm = l2
    step_length = step_lengths.get("l2", step_lengths["l1"])
    l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
    perturbation_l2 = torch.minimum(torch.tensor(1., dtype=x.dtype, device=x.device), gradients / l2norm)
    perturbation_l2[torch.isnan(perturbation_l2)] = 0.
    perturbation_l2[torch.isinf(perturbation_l2)] = 1.
    x_next_l2 = torch.clamp(x_next + perturbation_l2 * step_length, min=0., max=1.)
    x_adv_l2 = (((x_next_l2 - x_initial) >= 0) * x_next_l2) + (((x_next_l2 - x_initial) < 0) * x_initial)
    pertbx.append(x_adv_l2)
    # norm = l1
    step_length = step_lengths.get("l1", step_lengths["l1"])
    #ignore the gradient of indice which is updated
    gradients_l1 = gradients * (x_next < 0.5)
    val, _ = torch.topk(gradients_l1, 1)

    perturbation_l1 = torch.sign(gradients >= val.expand_as(gradients)) * (val > 1e-10)

    x_next_l1 = torch.clamp(x_next + perturbation_l1 * step_length, min=0., max=1.)
    x_adv_l1 = (((x_next_l1 - x_initial) >= 0) * x_next_l1) + (((x_next_l1 - x_initial) < 0) * x_initial)
    pertbx.append(x_adv_l1)

    return pertbx

In [ ]:
def StepwiseMax_onestep2(
    x,
    label,
    model,
    attack_list=["linf", "l2", "l1"],
    step_lengths={"l1": 1.0, "l2": 0.05, "linf": 0.002},
    steps=100,
    random_start=False,
    round_threshold=0.5,
):
  """
    Stepwise max attack (mixture of pgd-l1, pgd-l2, pgd-linf).

    Args:
        x: Input data tensor (shape: [batch_size, feature_dim])
        label: Ground truth labels tensor (shape: [batch_size])
        model: Victim model
        attack_list: List of attack norms (default: ["linf", "l2", "l1"])
        step_lengths: Dictionary mapping norm to its step length (default: {"l1": 1.0, "l2": 0.05, "linf": 0.002})
        steps: Maximum number of iterations (default: 100)
        random_start: Use random starting point (default: False)
        round_threshold: Threshold for rounding real scalars (default: 0.5)

    Returns:
        Adversarial examples tensor (same shape as x)
  """

  model.eval()

  n, red_n = x.shape
  adv_x = x.detach().clone()
  pert_x_cont = None
  prev_done = None
  for step in range(steps):
      with torch.no_grad():
          if step == 0 :
              adv_x = around_x(adv_x, std_deviation=0.1, random_start=random_start)

          _, done = get_loss(round_x(adv_x, round_threshold), label, model)

      if torch.all(done):
          break
      if step == 0:
          adv_x[~done] = x[~done]  # recompute the perturbation under other penalty factors
          prev_done = done
      else:
          adv_x[~done] = pert_x_cont[~done[~prev_done]]
          prev_done = done

      print('remaining samples : ',len(adv_x[~done]))
      num_sample_red = torch.sum(~done).item()

      pertbx = pgd_one_step2(adv_x[~done], label[~done], model, step_lengths,x[~done])

      with torch.no_grad():
          pertbx = torch.vstack(pertbx)

          n_attacks = len(attack_list)
          label_ext = torch.cat([label[~done]] * n_attacks)

          scores, _ = get_loss(pertbx, label_ext,model)
          _, _done = get_loss(round_x(pertbx, round_threshold), label_ext, model)

          max_v = scores.amax() if scores.amax() > 0 else 0.
          scores[_done] += max_v

          pertbx = pertbx.reshape(n_attacks, num_sample_red, red_n).permute([1, 0, 2])
          scores = scores.reshape(n_attacks, num_sample_red).permute(1, 0)
          _2, s_idx = scores.max(dim=-1)
          print('best attack : ',s_idx)
          pert_x_cont = pertbx[torch.arange(num_sample_red), s_idx]
          adv_x[~done] = pert_x_cont

  print(step)
  adv_x = round_x(adv_x, round_threshold)
  with torch.no_grad():
      _, done = get_loss(adv_x, label, model)
      print(f"step-wise max: attack effectiveness {done.sum().item() / done.size()[0] * 100:.3f}%.")

  return adv_x

In [ ]:
adv_x = StepwiseMax(X, y, model,step_lengths={"l1": 1.0, "l2": 0.05, "linf": 0.0008},steps=650,is_attacker=True)
print(X.sum()/batch_size)
print(adv_x.sum()/batch_size)

In [ ]:
adv_x = StepwiseMax2(X, y, model,step_lengths={"l1": 1.0, "l2": 0.05, "linf": 0.0008},steps=650,is_attacker=True)
print(X.sum()/batch_size)
print(adv_x.sum()/batch_size)

In [ ]:
adv_x = StepwiseMax_onestep2(X, y, model,step_lengths={"l1": 1.0, "l2": 0.05, "linf": 0.0008},steps=650)
print(X.sum()/batch_size)
print(adv_x.sum()/batch_size)

In [ ]:
print((adv_x - X).sum(dim=1))

## Step 10 — Mimicry on the full test set

In [ ]:
test_loader2 = DataLoader(test_dataset, batch_size=5000, shuffle=False)
X2,y2 = next(iter(test_loader2))
print(X2.shape)
print(y2.shape)

In [ ]:
adv_x = StepwiseMax_onestep2(X2[y2.squeeze()==1], y2[y2.squeeze()==1], model,step_lengths={"l1": 1.0, "l2": 0.1, "linf": 0.001},steps=300)
print(X2[y2.squeeze()==1].sum()/len(X2[y2.squeeze()==1]))
print(adv_x.sum()/len(X2[y2.squeeze()==1]))

In [ ]:
adv_x = StepwiseMax_onestep2(X2[y2.squeeze()==1], y2[y2.squeeze()==1], model,step_lengths={"l1": 1.0, "l2": 0.1, "linf": 0.001},steps=300,random_start=True)
print(X2[y2.squeeze()==1].sum()/len(X2[y2.squeeze()==1]))
print(adv_x.sum()/len(X2[y2.squeeze()==1]))

In [ ]:
ben_x = X2[y2.squeeze()==0]
ben_x.shape

In [ ]:
mal_x = X2[y2.squeeze()==1]
mal_x.shape

In [ ]:
trials = 100
seed = 200

trials = trials if trials < len(ben_x) else len(ben_x)
n= len(ben_x)//trials

with torch.no_grad():
    torch.manual_seed(seed)
    x_mod_list = []
    for _x in mal_x:
        indices = torch.randperm(len(ben_x))
        for i in range(n):
            selected = indices[(trials*i):(trials*(i+1))]
            trial_vectors = ben_x[selected]
            modified_x = torch.clamp(_x + trial_vectors, min=0., max=1.) #Size([trials, 10000])
            _,done = get_loss(modified_x,torch.ones(trials,1),model)
            if done.sum() > 0:
                added_features = (modified_x - _x).sum(dim=1)
                arg_min = added_features[done].argmin()
                adv_x = modified_x[done][arg_min].unsqueeze(0)
                x_mod_list.append(adv_x)
                break
            else:
                continue

print(f"mimic attack effectiveness {len(x_mod_list) / len(mal_x) * 100:.3f}%.")

In [ ]:
def mimicry(ben_x, mal_x, model, trials, seed):
    trials = min(trials, len(ben_x))
    n = len(ben_x) // trials
    x_mod_list = []

    with torch.no_grad():
        torch.manual_seed(seed)
        for _x in mal_x:
            indices = torch.randperm(len(ben_x))
            for i in range(n):
                selected = indices[(trials * i):(trials * (i + 1))]
                trial_vectors = ben_x[selected]
                modified_x = torch.clamp(_x + trial_vectors, min=0., max=1.)
                _, done = get_loss(modified_x, torch.ones(trials, 1), model)
                if done.sum() > 0:
                    added_features = (modified_x - _x).sum(dim=1)
                    arg_min = added_features[done].argmin()
                    adv_x = modified_x[done][arg_min].unsqueeze(0)
                    x_mod_list.append(adv_x)
                    break

    attack_effectiveness = len(x_mod_list) / len(mal_x) * 100
    print(f"Mimic attack effectiveness: {attack_effectiveness:.3f}%.")
    return torch.vstack(x_mod_list)

In [ ]:
adv = mimicry(ben_x, mal_x, model, trials=1000, seed=10)
print(mal_x.sum()/len(mal_x))
print(adv.sum()/len(mal_x))

In [ ]:
adv = mimicry(ben_x, mal_x, model, trials=2000, seed=10)
print(mal_x.sum()/len(mal_x))
print(adv.sum()/len(mal_x))